## Imports

In [1]:
!pip install -q kagglehub

In [2]:
import os
import json
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

import matplotlib.pyplot as plt

import kagglehub

## Upload data

In [3]:
path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

print(path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


## Read Files

In [4]:
datasets = {}

for file in os.listdir(path):

    if file.endswith(".csv"):

        name = file.replace(".csv", "")

        datasets[name] = pd.read_csv(
            os.path.join(path, file)
        )

for name, df in datasets.items():
    print(name, df.shape)

olist_customers_dataset (99441, 5)
olist_sellers_dataset (3095, 4)
olist_order_reviews_dataset (99224, 7)
olist_order_items_dataset (112650, 7)
olist_products_dataset (32951, 9)
olist_geolocation_dataset (1000163, 5)
product_category_name_translation (71, 2)
olist_orders_dataset (99441, 8)
olist_order_payments_dataset (103886, 5)


## Data Cleaning
remove duplicates and convert datetime

In [5]:
# Remove exact duplicates
for name in datasets:
    datasets[name] = datasets[name].drop_duplicates()

# Convert order dates
orders = datasets["olist_orders_dataset"]

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

print("Data cleaning completed.")

Data cleaning completed.


## Build Business Dataset
concat relatable data

In [6]:
customers = datasets["olist_customers_dataset"]
order_items = datasets["olist_order_items_dataset"]
products = datasets["olist_products_dataset"]
sellers = datasets["olist_sellers_dataset"]
payments = datasets["olist_order_payments_dataset"]
reviews = datasets["olist_order_reviews_dataset"]
categories = datasets["product_category_name_translation"]

In [7]:
business_data = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

business_data = business_data.merge(
    order_items,
    on="order_id",
    how="left"
)

business_data = business_data.merge(
    products,
    on="product_id",
    how="left"
)

business_data = business_data.merge(
    sellers,
    on="seller_id",
    how="left"
)

business_data = business_data.merge(
    categories,
    on="product_category_name",
    how="left"
)

print(business_data.shape)

(113425, 30)


## Business Analysis


to get Business KPIs

In [8]:
total_revenue = payments["payment_value"].sum()

total_orders = orders["order_id"].nunique()

total_customers = customers["customer_unique_id"].nunique()

average_order_value = (
    total_revenue / total_orders
)

print("Revenue:", round(total_revenue, 2))
print("Orders:", total_orders)
print("Customers:", total_customers)
print("Average Order Value:", round(average_order_value, 2))

Revenue: 16008872.12
Orders: 99441
Customers: 96096
Average Order Value: 160.99


Get Product Categories

In [9]:
category_sales = (
    business_data
    .groupby("product_category_name_english")
    ["price"]
    .sum()
    .sort_values(ascending=False)
)

print("Top Categories:")
display(category_sales.head(10))

Top Categories:


,price
product_category_name_english,
health_beauty,1258681.34
watches_gifts,1205005.68
bed_bath_table,1036988.68
sports_leisure,988048.97
computers_accessories,911954.32
furniture_decor,729762.49
cool_stuff,635290.85
housewares,632248.66
auto,592720.11


Customer Segmentation (Using K-means)

In [10]:
customer_orders = orders.merge(
    customers[
        ["customer_id", "customer_unique_id"]
    ],
    on="customer_id"
)

customer_spending = (
    payments
    .merge(
        customer_orders[
            ["order_id", "customer_unique_id"]
        ],
        on="order_id"
    )
    .groupby("customer_unique_id")
    .agg(
        spending=("payment_value", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

customer_spending.head()

,customer_unique_id,spending,orders
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,1
2,0000f46a3911fa3c0805444483337064,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,196.89,1


In [11]:
X = customer_spending[
    ["spending", "orders"]
]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

customer_spending["segment"] = model.fit_predict(
    X_scaled
)

customer_spending.head()

,customer_unique_id,spending,orders,segment
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,1,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,1,0
2,0000f46a3911fa3c0805444483337064,86.22,1,0
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,1,0
4,0004aac84e0df4da2b147fca70cf8255,196.89,1,0


Sales Anomaly Detection using Isolation Forest Model (-1 means anomaly)

In [12]:
orders["date"] = (
    orders["order_purchase_timestamp"].dt.date
)

daily_sales = (
    payments
    .merge(
        orders[["order_id", "date"]],
        on="order_id"
    )
    .groupby("date")["payment_value"]
    .sum()
    .reset_index()
)

model = IsolationForest(
    contamination=0.03,
    random_state=42
)

daily_sales["anomaly"] = model.fit_predict(
    daily_sales[["payment_value"]]
)

anomalies = daily_sales[
    daily_sales["anomaly"] == -1
]

display(anomalies)

,date,payment_value,anomaly
16,2017-01-07,1447.99,-1
23,2017-01-14,1507.36,-1
337,2017-11-24,179200.03,-1
338,2017-11-25,71897.01,-1
339,2017-11-26,53939.87,-1
340,2017-11-27,56966.28,-1
341,2017-11-28,56708.41,-1
347,2017-12-04,56225.41,-1
473,2018-04-09,55782.82,-1
501,2018-05-07,62568.25,-1


In [13]:
os.makedirs("agent_data", exist_ok=True)

business_data.to_csv(
    "agent_data/business_data.csv",
    index=False
)

customer_spending.to_csv(
    "agent_data/customer_segments.csv",
    index=False
)

daily_sales.to_csv(
    "agent_data/daily_sales.csv",
    index=False
)

anomalies.to_csv(
    "agent_data/sales_anomalies.csv",
    index=False
)

kpis = {
    "total_revenue": float(total_revenue),
    "total_orders": int(total_orders),
    "total_customers": int(total_customers),
    "average_order_value": float(average_order_value)
}

with open(
    "agent_data/kpis.json",
    "w"
) as f:

    json.dump(
        kpis,
        f,
        indent=4
    )

print("All outputs saved successfully.")

All outputs saved successfully.


In [14]:
print("Files ready for Notebook 2:")

for file in os.listdir("agent_data"):
    print(file)

Files ready for Notebook 2:
business_data.csv
daily_sales.csv
sales_anomalies.csv
kpis.json
customer_segments.csv


zip agent_data for download

In [15]:
import shutil

shutil.make_archive(
    "agent_data",
    "zip",
    "agent_data"
)

print("agent_data.zip created successfully!")

agent_data.zip created successfully!


Check Data

In [16]:
import os
import json
import pandas as pd

required_files = [
    "business_data.csv",
    "customer_segments.csv",
    "daily_sales.csv",
    "sales_anomalies.csv",
    "kpis.json"
]

print("Checking agent_data...\n")

for file in required_files:
    path = os.path.join("agent_data", file)

    if os.path.exists(path):
        print(f"✅ {file}")
    else:
        print(f"❌ {file} is missing")

print("\nLoading data...")

business_data = pd.read_csv("agent_data/business_data.csv")
customer_segments = pd.read_csv("agent_data/customer_segments.csv")
daily_sales = pd.read_csv("agent_data/daily_sales.csv")
sales_anomalies = pd.read_csv("agent_data/sales_anomalies.csv")

with open("agent_data/kpis.json", "r") as f:
    kpis = json.load(f)

print("\nData loaded successfully!")

print("\nShapes:")
print("Business data:", business_data.shape)
print("Customer segments:", customer_segments.shape)
print("Daily sales:", daily_sales.shape)
print("Sales anomalies:", sales_anomalies.shape)

print("\nKPIs:")
print(kpis)

Checking agent_data...

✅ business_data.csv
✅ customer_segments.csv
✅ daily_sales.csv
✅ sales_anomalies.csv
✅ kpis.json

Loading data...

Data loaded successfully!

Shapes:
Business data: (113425, 30)
Customer segments: (96095, 4)
Daily sales: (633, 3)
Sales anomalies: (19, 3)

KPIs:
{'total_revenue': 16008872.12, 'total_orders': 99441, 'total_customers': 96096, 'average_order_value': 160.98864774087147}
